# Vast AI Dataset Inspection

This notebook is for downloading the Kaggle dataset and performing dataset performance analysis only. It does not connect to the project training scripts.

In [ ]:
import os
import sys

print("Python executable:", sys.executable)
print("Current working directory:", os.getcwd())
print("Contents:", os.listdir(os.getcwd()))

## Install required packages

Install Kaggle API and analysis libraries for dataset inspection.

In [ ]:
import sys
!"{sys.executable}" -m pip install -q kaggle pandas pillow matplotlib seaborn

## Configure Kaggle credentials

Upload `kaggle.json` to the server if it is not already available. The notebook will use `~/.kaggle/kaggle.json`.

In [ ]:
import os
from pathlib import Path
import shutil

project_root = Path.cwd()
local_kaggle_file = project_root / "kaggle.json"
kaggle_dir = Path.home() / ".kaggle"
kaggle_dir.mkdir(parents=True, exist_ok=True)
cred_path = kaggle_dir / "kaggle.json"

print("Project root:", project_root)
print("Local kaggle.json path:", local_kaggle_file)
print("Kaggle config path:", cred_path)

if local_kaggle_file.exists():
    shutil.copy(local_kaggle_file, cred_path)
    os.chmod(cred_path, 0o600)
    print("Copied kaggle.json from project root to ~/.kaggle/kaggle.json")
elif cred_path.exists():
    os.chmod(cred_path, 0o600)
    print("Using existing ~/.kaggle/kaggle.json")
else:
    print("Please upload kaggle.json to the project root or ~/.kaggle/kaggle.json.")
    print("Current working directory contents:", sorted([p.name for p in project_root.iterdir()]))

## Download dataset from Kaggle

This uses the Kaggle dataset `ayushmandatta1/deepdetect-2025`. Update the dataset slug if you need a different dataset.

In [ ]:
import os
from pathlib import Path

download_root = Path(os.getcwd()) / "data" / "kaggle_dataset"
download_root.mkdir(parents=True, exist_ok=True)

print("Download root:", download_root)

!kaggle datasets download -d ayushmandatta1/deepdetect-2025 -p "{download_root}" --unzip

print("Downloaded files:")
!ls -la "{download_root}"

## Locate dataset root

The Kaggle archive may extract into a subfolder. Detect the dataset root automatically.

In [ ]:
from pathlib import Path

dataset_root = None
for child in sorted((download_root).iterdir()):
    if child.is_dir() and any(child.glob("**/*.jpg")):
        dataset_root = child
        break

if dataset_root is None:
    # default to direct download_root if it already has train/val/test
    if (download_root / "ddata").exists():
        dataset_root = download_root / "ddata"
    else:
        dataset_root = download_root

print("Dataset root:", dataset_root)
print("Has train:", (dataset_root / "train").exists())
print("Has val:", (dataset_root / "val").exists())
print("Has test:", (dataset_root / "test").exists())

## Basic dataset structure summary

Count files in each split and class folder.

In [ ]:
import os
from pathlib import Path

splits = ["train", "val", "test"]
classes = ["real", "fake"]
root = dataset_root

summary = []
for split in splits:
    for cls in classes:
        folder = root / split / cls
        if folder.exists():
            count = sum(1 for _ in folder.glob("**/*") if _.is_file())
        else:
            count = 0
        summary.append((split, cls, count, str(folder)))

for split, cls, count, path in summary:
    print(f"{split}/{cls}: {count} files -> {path}")

## Create validation split if missing

If the dataset has no validation images, move 20% of each training class into `val`.

In [ ]:
import random
import shutil
from pathlib import Path

val_root = root / "val"
val_file_count = sum(1 for _ in val_root.rglob("**/*") if _.is_file()) if val_root.exists() else 0

if val_file_count == 0:
    print("Validation set is missing or empty. Creating validation split from train.")
    for cls in classes:
        train_cls = root / "train" / cls
        val_cls = val_root / cls
        val_cls.mkdir(parents=True, exist_ok=True)
        images = [p for p in sorted(train_cls.iterdir()) if p.is_file()]
        random.shuffle(images)
        num_val = int(len(images) * 0.2)
        moved = 0
        for src_path in images[:num_val]:
            dest_path = val_cls / src_path.name
            shutil.move(str(src_path), str(dest_path))
            moved += 1
        print(f"Moved {moved} images from train/{cls} to val/{cls}")
else:
    print(f"Validation set already exists with {val_file_count} files.")

## Dataset inspection and statistics

Analyze image size, file type, corrupted files, and class balance.

In [ ]:
import hashlib
from collections import defaultdict
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image, ImageFile, UnidentifiedImageError

ImageFile.LOAD_TRUNCATED_IMAGES = True

valid_exts = {".jpg", ".jpeg", ".png", ".bmp", ".webp", ".tif", ".tiff"}
records = []
corrupted = []
hash_groups = defaultdict(list)

for split in splits:
    for cls in classes:
        folder = root / split / cls
        if not folder.exists():
            continue

        for path in sorted(folder.rglob("*")):
            if not path.is_file():
                continue
            if path.suffix.lower() not in valid_exts:
                continue

            entry = {
                "split": split,
                "class": cls,
                "path": str(path),
                "filename": path.name,
                "ext": path.suffix.lower(),
                "size_bytes": path.stat().st_size,
                "width": None,
                "height": None,
                "aspect_ratio": None,
                "corrupted": False,
            }

            try:
                with Image.open(path) as img:
                    img.verify()
                with Image.open(path) as img:
                    img = img.convert("RGB")
                    width, height = img.size
                    entry["width"] = width
                    entry["height"] = height
                    entry["aspect_ratio"] = round(width / height, 4) if height else None

                file_hash = hashlib.md5(path.read_bytes()).hexdigest()
                hash_groups[file_hash].append(str(path))

            except (UnidentifiedImageError, OSError, ValueError) as exc:
                entry["corrupted"] = True
                corrupted.append((str(path), str(exc)))

            records.append(entry)

metadata = pd.DataFrame(records)
print("Total scanned files:", len(metadata))
print("Corrupted files:", metadata["corrupted"].sum())

if len(corrupted) > 0:
    print("\nCorrupted file examples:")
    for path, err in corrupted[:5]:
        print(path, err)

print("\nDuplicate file groups:")
duplicate_groups = [paths for paths in hash_groups.values() if len(paths) > 1]
print("Number of duplicate groups:", len(duplicate_groups))
if duplicate_groups:
    for idx, group in enumerate(duplicate_groups[:3], start=1):
        print(f"Group {idx}: {len(group)} files")
        for p in group[:3]:
            print("  -", p)

metadata.to_csv("dataset_inspection.csv", index=False)
print("\nSaved dataset_inspection.csv")

## Statistics summary

In [ ]:
print("Class balance by split:")
print(metadata.groupby(["split", "class"]).size().unstack(fill_value=0))

print("\nOverall counts:")
print(metadata["class"].value_counts())

print("\nImage size stats:")
print(metadata[["width", "height", "aspect_ratio"]].describe())

print("\nFile type counts:")
print(metadata["ext"].value_counts())

## Visualize dataset properties

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.set(style="whitegrid")

fig, ax = plt.subplots(figsize=(8, 5))
metadata.groupby(["split", "class"]).size().unstack().plot(kind="bar", ax=ax)
ax.set_title("Image count by split and class")
ax.set_ylabel("Number of images")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(8, 5))
metadata[metadata["corrupted"] == False]["aspect_ratio"].hist(bins=40, ax=ax)
ax.set_title("Aspect Ratio Distribution")
ax.set_xlabel("Width / Height")
ax.set_ylabel("Number of images")
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(8, 5))
metadata[metadata["corrupted"] == False]["width"].hist(bins=40, alpha=0.6, label="width", ax=ax)
metadata[metadata["corrupted"] == False]["height"].hist(bins=40, alpha=0.6, label="height", ax=ax)
ax.set_title("Image Dimension Distribution")
ax.set_xlabel("Pixels")
ax.set_ylabel("Frequency")
ax.legend()
plt.tight_layout()
plt.show()